# 14.4 · ARIMA / SARIMA / SARIMAX ⭐

> **课程定位 / Where this fits**
> 第 4 课，**Part 14 · 时间序列**。经典统计预测的"集大成者", 时序面试的核心。
> Lesson 4, **Part 14 · Time Series**. The crown jewel of classical forecasting and the core of TS interviews.
>
> **ARIMA** 把三个想法合在一起:**AR(自回归)** 用过去的值预测、**I(差分)** 去趋势使平稳、**MA(移动平均)** 用过去的预测误差。**SARIMA** 再加上**季节项**(处理周期);**SARIMAX** 再加上**外生变量**(如促销、天气等额外信息)。它是数十年来预测的标准工具, 至今仍是强基线。本课讲清每个字母的含义、用 ACF/PACF 和 AIC **定阶**、在航空数据上建模预测并做**残差诊断**。
> **ARIMA** combines three ideas: **AR (autoregression)** predicts from past values, **I (integration/differencing)** removes trend for stationarity, **MA (moving average)** uses past forecast errors. **SARIMA** adds **seasonal terms**; **SARIMAX** adds **exogenous variables** (promotions, weather, etc.). The standard forecasting tool for decades, still a strong baseline. We explain each letter, **select orders** with ACF/PACF and AIC, forecast the airline data, and do **residual diagnostics**.
>
> 💼 **实战/面试视角**：ARIMA 是时序面试**绝对核心**——"AR/I/MA各是什么 / pdq怎么定 / SARIMA的季节项 / 残差诊断 / ARIMA vs 指数平滑" 几乎必考。
> 💼 **Practical/interview angle:** ARIMA is **central** to TS interviews — "what AR/I/MA are / choosing p,d,q / SARIMA's seasonal terms / residual diagnostics / ARIMA vs smoothing" — almost guaranteed.

> 📐 **符号约定 / Notation**
> - ARIMA(p,d,q):AR阶 p, 差分阶 d, MA阶 q / AR order p, differencing d, MA order q
> - SARIMA(p,d,q)(P,D,Q,s):后者是季节阶, s 是周期 / seasonal orders + period s

> 💡 **面试相关 / Interview-relevant**
> - "ARIMA 的 AR/I/MA 分别是什么"（出镜率 ★★★★★）
> - "p,d,q 怎么确定(ACF/PACF/AIC)"（★★★★★）
> - "SARIMA 的季节项 (P,D,Q,s)"（★★★★）
> - "残差诊断要看什么(白噪声)"（★★★★）
> - "SARIMAX 的外生变量"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解 AR / I / MA 三个成分各自的含义。
   Understand the AR / I / MA components.
2. 用 ACF/PACF 和 **AIC** 给模型**定阶**。
   Select orders via ACF/PACF and AIC.
3. 用 **SARIMA** 在航空数据上预测并诊断残差。
   Forecast with SARIMA and diagnose residuals.
4. 理解 **SARIMAX** 的外生变量与 auto_arima。
   Understand SARIMAX's exogenous variables and auto_arima.

## 目录 / TOC
1. [ARIMA = AR + I + MA ⭐](#1)
2. [定阶:ACF/PACF 与 AIC ⭐](#2)
3. [SARIMA 预测 + 残差诊断 ⭐](#3)
4. [SARIMAX 与 auto_arima + 小结 ⭐](#4)


<a id="1"></a>
## 1. ARIMA = AR + I + MA ⭐ / ARIMA Components

**ARIMA(p, d, q)** 由三个部分组成(面试必背)：
**ARIMA(p, d, q)** has three parts (must-know):
- **AR(p) 自回归**:用**过去 $p$ 个观测值**的线性组合预测当前值。"今天≈昨天和前天的加权"。$p$ = 用几个历史值。
  **AR(p) autoregression:** predict the current value from a linear combination of the **past $p$ observations**. "Today ≈ weighted yesterday & before." $p$ = how many lags.
- **I(d) 差分**:对序列做 $d$ 次差分使其**平稳**(14.1/14.3 见过)。$d$ = 差分几次。
  **I(d) integration:** difference the series $d$ times to make it **stationary** (seen in 14.1/14.3). $d$ = how many differences.
- **MA(q) 移动平均**:用**过去 $q$ 个预测误差**的线性组合修正。"根据最近几次预测偏差来纠正"。$q$ = 用几个历史误差。
  **MA(q) moving average:** correct using a linear combination of the **past $q$ forecast errors**. "Adjust by recent forecast mistakes." $q$ = how many error lags.

⚠️ 注意: 这里的 **MA(移动平均)** 指"过去**误差**的平均", 和 14.3 的移动平均(过去**值**的平均)**是两回事**——同名不同义, 是常见混淆点(面试坑)。
⚠️ Note: this **MA** means "average of past **errors**," NOT 14.3's moving average (of past **values**) — same name, different meaning, a common confusion (interview trap).

合起来: ARIMA 先**差分**到平稳, 再用**自回归 + 误差修正**建模。它要求序列(差分后)平稳, 适合有趋势但**无强季节**的数据;有季节就要 SARIMA。
Together: ARIMA **differences** to stationarity, then models with **autoregression + error correction**. It requires (differenced) stationarity and suits trended but **non-strongly-seasonal** data; for seasonality use SARIMA.


<a id="2"></a>
## 2. 定阶:ACF/PACF 与 AIC ⭐ / Order Selection

怎么确定 $p, d, q$(面试高频)? 两种思路结合：
How to choose $p, d, q$ (high-frequency)? Combine two approaches:
- **$d$(差分阶)**:做几次差分能让序列平稳(用 ADF 检验, 14.1)。通常 1 次足够;有季节再加季节差分。
  **$d$:** how many differences make it stationary (ADF test, 14.1). Usually 1; add seasonal differencing if seasonal.
- **$p, q$ 看 ACF/PACF**(14.1 提过的经验规则)：**PACF 在滞后 $p$ 后截断 → AR(p)**;**ACF 在滞后 $q$ 后截断 → MA(q)**。
  **$p, q$ from ACF/PACF** (the rule from 14.1): **PACF cuts off after lag $p$ → AR(p)**; **ACF cuts off after lag $q$ → MA(q)**.
- **AIC(赤池信息准则)**:更客观的自动办法——**网格搜索**多组 $(p,q)$, 选 **AIC 最小**的(AIC 在"拟合好"和"模型简单"间平衡, 防过拟合)。这就是 `auto_arima` 的核心思路。
  **AIC (Akaike Information Criterion):** a more objective automatic way — **grid-search** several $(p,q)$ and pick the **lowest AIC** (AIC balances fit vs simplicity, avoiding overfit). The core idea of `auto_arima`.

下面用 **AIC 网格搜索**自动定阶(实现 auto_arima 的思想)。
Below we use **AIC grid search** to select orders (implementing auto_arima's idea).


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, itertools, warnings
warnings.filterwarnings("ignore")
from statsmodels.tsa.statespace.sarimax import SARIMAX
sns.set_theme(style="whitegrid")
ap = [112,118,132,129,121,135,148,148,136,119,104,118, 115,126,141,135,125,149,170,170,158,133,114,140,
      145,150,178,163,172,178,199,199,184,162,146,166, 171,180,193,181,183,218,230,242,209,191,172,194,
      196,196,236,235,229,243,264,272,237,211,180,201, 204,188,235,227,234,264,302,293,259,229,203,229,
      242,233,267,269,270,315,364,347,312,274,237,278, 284,277,317,313,318,374,413,405,355,306,271,306,
      315,301,356,348,355,422,465,467,404,347,305,336, 340,318,362,348,363,435,491,505,404,359,310,337,
      360,342,406,396,420,472,548,559,463,407,362,405, 417,391,419,461,472,535,622,606,508,461,390,432]
idx = pd.date_range("1949-01", periods=len(ap), freq="MS"); ts = pd.Series(ap, index=idx, name="passengers")

train, test = ts[:120], ts[120:]                          # 按时间切: 前10年训练, 后2年测试 / chronological split
# AIC 网格搜索定阶(auto_arima思想): 试几组季节SARIMA, 选AIC最小 / grid search by AIC
results = []
for p, q, P, Q in itertools.product([0,1], [0,1], [0,1], [0,1]):
    try:
        m = SARIMAX(np.log(train), order=(p,1,q), seasonal_order=(P,1,Q,12)).fit(disp=False)
        results.append((m.aic, (p,1,q), (P,1,Q,12)))      # 记录 AIC 和阶数 / record AIC and orders
    except Exception: pass
results.sort()                                            # 按 AIC 升序 / sort by AIC ascending
print("AIC 网格搜索(选最小的) — 前5个候选:")
for aic, o, so in results[:5]:
    print(f"  SARIMA{o}{so}: AIC = {aic:.1f}")
best_order, best_sorder = results[0][1], results[0][2]
print(f"\n最佳(AIC最小): SARIMA{best_order}{best_sorder}")
print("AIC 在'拟合好'和'参数少'间权衡, 自动选阶防过拟合 → 这就是 auto_arima 干的事")


<a id="3"></a>
## 3. SARIMA 预测 + 残差诊断 ⭐ / SARIMA Forecast & Diagnostics

**SARIMA(p,d,q)(P,D,Q,s)** 在 ARIMA 上加了**季节部分**:$(P,D,Q)$ 是**季节的 AR/差分/MA 阶**, $s$ 是**季节周期**(月度年度季节 $s=12$)。它能同时建模"短期依赖"和"季节性依赖"——非常适合 AirPassengers。
**SARIMA(p,d,q)(P,D,Q,s)** adds a **seasonal part** to ARIMA: $(P,D,Q)$ are the **seasonal AR/differencing/MA orders**, $s$ the **seasonal period** ($s=12$ for monthly-yearly). It models both short-term and seasonal dependence — perfect for AirPassengers.

用上面选出的最佳阶数拟合、预测最后 2 年, 并和真实值比较。
We fit the best orders, forecast the last 2 years, and compare to truth.


In [ ]:
best = SARIMAX(np.log(train), order=best_order, seasonal_order=best_sorder).fit(disp=False)
# 预测(在log尺度), 再 exp 回原尺度 / forecast in log, then exp back
pred = best.get_forecast(len(test)); fc = np.exp(pred.predicted_mean)
ci = np.exp(pred.conf_int())                              # 预测置信区间 / confidence interval
mape = (np.abs((test.values - fc.values)/test.values)).mean()*100
fig, ax = plt.subplots(figsize=(11, 4))
train.plot(ax=ax, label="训练"); test.plot(ax=ax, label="真实", color="green")
fc.plot(ax=ax, label="SARIMA 预测", color="red", ls="--")
ax.fill_between(ci.index, ci.iloc[:,0], ci.iloc[:,1], color="red", alpha=0.15, label="95%置信区间")
ax.legend(); ax.set_title(f"SARIMA{best_order}{best_sorder} 预测后2年: MAPE={mape:.1f}% (附预测不确定性区间)")
plt.tight_layout(); plt.show()
print(f"SARIMA 预测 MAPE = {mape:.1f}%; SARIMA 还能给出'预测置信区间'(指数平滑/ML常没有) → 量化不确定性")


In [ ]:
# 残差诊断: 好模型的残差应是'白噪声'(无自相关、近似正态) / residual diagnostics: should be white noise
fig = best.plot_diagnostics(figsize=(12, 7)); plt.tight_layout(); plt.show()
from statsmodels.stats.diagnostic import acorr_ljungbox
lb = acorr_ljungbox(best.resid[1:], lags=[12], return_df=True)   # Ljung-Box 检验残差是否还有自相关 / Ljung-Box test
p_lb = lb["lb_pvalue"].iloc[0]
print(f"Ljung-Box 检验(lag12) p值 = {p_lb:.3f}  {'→ 残差无显著自相关(像白噪声, 模型抓住了规律) ✓' if p_lb>0.05 else '→ 残差仍有自相关(模型没抓干净)'}")
print("残差诊断要看(面试): ①残差像白噪声(无自相关→Ljung-Box p>0.05) ②近似正态(QQ图) ③ACF落在置信带内")
print("若残差还有结构 → 模型没抓住全部规律, 需调整阶数")


<a id="4"></a>
## 4. SARIMAX 与 auto_arima + 小结 ⭐ / SARIMAX & auto_arima

**SARIMAX** 里的 **X = eXogenous(外生变量)**:除了序列自身的历史, 还能加入**外部信息**作为额外的预测变量——比如预测销量时加入"是否促销""节假日""天气""油价"等。模型同时利用"自身规律(SARIMA 部分)+ 外部因素(X 部分)"。
The **X in SARIMAX = eXogenous variables:** beyond the series' own history, add **external information** as extra predictors — e.g. forecasting sales with "is there a promotion," "holiday," "weather," "oil price." The model uses both "self-dynamics (SARIMA part) + external factors (X part)."

> ⚠️ 实战坑(面试): 用外生变量预测**未来**时, 你必须**知道未来的外生变量值**(或也得预测它)。"明天会不会促销"你知道(可控), 但"明天的油价"你不知道——用了未知的未来外生变量就是**泄漏**。
> ⚠️ Pitfall: to forecast the **future** with exogenous variables, you must **know their future values** (or forecast them too). "Will there be a promo tomorrow" you know (controllable), but "tomorrow's oil price" you don't — using unknown future exogenous = leakage.

**auto_arima**(`pmdarima` 库): 自动做我们 §2 的事——网格/逐步搜索 $p,d,q,P,D,Q$, 按 AIC 选最优, 省去手动定阶。实务常用它快速起步。
**auto_arima** (`pmdarima` library): automates §2 — grid/stepwise search over orders, picks by AIC, sparing manual tuning. Commonly used to start quickly.

下面演示 SARIMAX 加一个外生变量(这里用一个合成的"是否旺季"指示, 它在未来可知)。
Below we demo SARIMAX with an exogenous variable (a synthetic "peak-season" indicator, known in the future).


In [ ]:
# 构造一个外生变量: 暑期旺季指示(6,7,8月=1), 未来可知 / exogenous: summer-peak indicator (known in future)
exog = pd.Series((ts.index.month.isin([7,8])).astype(float), index=ts.index, name="summer_peak")
exog_tr, exog_te = exog[:120], exog[120:]
sx = SARIMAX(np.log(train), exog=exog_tr, order=best_order, seasonal_order=best_sorder).fit(disp=False)
fc_x = np.exp(sx.get_forecast(len(test), exog=exog_te).predicted_mean)   # 预测时提供未来的外生变量 / future exog provided
mape_x = (np.abs((test.values - fc_x.values)/test.values)).mean()*100
print(f"SARIMA (无外生)  预测 MAPE = {mape:.1f}%")
print(f"SARIMAX(加旺季指示) 预测 MAPE = {mape_x:.1f}%")
print("(本例季节已被SARIMA的季节项抓住, 额外旺季指示提升有限; 真正有用的是模型未知的外部信息如促销/天气)")
print("\nauto_arima(pmdarima库)自动搜p,d,q,P,D,Q并按AIC选最优, 实务常用; 本课用AIC网格搜索演示了同样思想")


```
ARIMA(p,d,q): AR(p)用过去p个值 + I(d)差分d次使平稳 + MA(q)用过去q个预测误差
注意: ARIMA的MA(过去误差均值) ≠ 14.3移动平均(过去值均值), 同名不同义(面试坑)
定阶: d看ADF(差几次平稳); p,q看ACF/PACF(PACF p后截断→AR(p), ACF q后截断→MA(q)); 或AIC网格搜索(=auto_arima)
SARIMA(p,d,q)(P,D,Q,s): 加季节AR/差分/MA阶 + 周期s(月度s=12); 同时建模短期+季节依赖
残差诊断: 好模型残差=白噪声(Ljung-Box p>0.05无自相关 + QQ正态 + ACF落置信带); 否则没抓干净
SARIMAX: X=外生变量(促销/天气等外部信息); 预测未来需知道未来的外生值(否则泄漏)
SARIMA还给预测置信区间(量化不确定性); auto_arima自动定阶
```

### 💡 面试速查 / Interview cheat-sheet
1. **AR/I/MA**: AR用过去值, I差分使平稳, MA用过去误差; ARIMA(p,d,q)。
   AR/I/MA: AR past values, I differencing for stationarity, MA past errors.
2. **定阶**: d看ADF; p,q看PACF/ACF截断; 或AIC网格搜索(auto_arima)。
   Orders: d via ADF; p,q via PACF/ACF cutoffs; or AIC grid search.
3. **SARIMA**: 加季节(P,D,Q,s); 处理周期性(s=周期)。
   SARIMA: adds seasonal (P,D,Q,s) for periodicity.
4. **残差诊断**: 应为白噪声(无自相关Ljung-Box/正态QQ); 否则没抓干净。
   Diagnostics: residuals should be white noise; else underfit.
5. **SARIMAX**: 加外生变量; 预测需知未来外生值(防泄漏); ARIMA给置信区间。
   SARIMAX: exogenous vars; need their future values (avoid leakage); ARIMA gives intervals.

### 下一节 / Next
**14.5 Prophet 思想**——Facebook 开源的预测工具, 思路和 ARIMA 完全不同: 把序列建模成**可解释的可加成分(趋势 + 季节 + 节假日)**, 用回归拟合, 对缺失/异常/多季节稳健、易调。我们会**从零实现这套可加模型的核心思想**(趋势 + Fourier 季节)。
**14.5 Prophet** — Facebook's open-source tool, very different from ARIMA: model the series as **interpretable additive components (trend + seasonality + holidays)**, fit by regression, robust to missing/outliers/multiple seasons and easy to tune. We'll **implement the core additive-model idea from scratch** (trend + Fourier seasonality).
